# Lecture 8 - LLMs and agent harnesses

**Lecture Topic**: How large language models work from a user's point of view, and what the harness around them (the "agent" part) adds

## Announcements

- Project proposals (Milestone 1) are due this Friday, Oct 16 at 11:59pm. Please let me know by Wednesday if you're still looking for a group.
- Wednesday's lecture is also on Zoom, and there is no class on Friday.
- ClassBuzz poll for today: `C5D9`

## Recap

In the last lecture, we covered the following topics:
 - Why we visualize data
 - matplotlib figures, axes and subplots (and why not to mix `plt.` and `ax.`)
 - Plotting with pandas
 - xarray DataArrays and Datasets: dimensions, coordinates and attributes
 - Indexing with `sel` and `isel`
 - Weighting

Today, we will cover the following topics:
 - Finishing xarray: broadcasting by name, alignment and groupby
 - What an LLM actually does: tokens, probabilities and sampling
 - Training, knowledge cutoffs and open-weights models
 - Chats, context windows and tool calling
 - The anatomy of a harness, and a real Claude Code session traced step by step

## Finishing off xarray

Let's reload the temperature data and the latitude weights from last time:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

ds = xr.open_dataset('../data/global_temparature.nc').drop_vars('quantile')
weights = np.cos(np.deg2rad(ds.lat))

### Broadcasting by name

`weights` has one dimension (`lat`) and `ds.tas` has three (`time`, `lat`, `lon`). Xarray broadcasts by dimension *name*, so there's no need to worry about the order of the axes or adding size-1 dimensions like we did with `keepdims` in NumPy:

In [ ]:
global_mean = ds.tas.weighted(weights).mean(['lat', 'lon'])

fig, ax = plt.subplots(figsize=(6, 3))
global_mean.plot(ax=ax, label='weighted')
ds.tas.mean(['lat', 'lon']).plot(ax=ax, label='unweighted')
ax.legend()

## High level computation

Xarray has very similar tools to pandas for common computations: `groupby`, `resample`, `rolling` and `coarsen`. For example, we can group by the country each grid cell is in:

In [ ]:
country = xr.open_dataarray('../data/country_mask.nc').rename('country')

warming_by_country = ds.tas.sel(time=2100).groupby(country).mean()
warming_by_country.to_series().nlargest(5)

Note, `to_series()` (and `to_dataframe()`) convert to pandas. And to read and write files, `xr.open_dataset()` and `ds.to_netcdf()` work much like `pd.read_csv()` and `df.to_csv()`.

That's the end of Module A!

# Module B: Using coding agents

For the next few lectures we'll look at how coding agents actually work, and how to steer and check them.

Today is about the machinery. We won't go into how the neural networks are built or trained in any detail (there are other DSC courses for that). Instead we'll focus on the parts that change how an agent *behaves*, since that is what you need to know to use one well.

An agent like Claude Code, Codex or Copilot is two things:
 - a **model** (the LLM), which only ever produces text
 - a **harness**, the ordinary program around it that sends the model text, runs the tools it asks for, and decides what it's allowed to do

## An LLM predicts the next token

At its core, a large language model (LLM) is a function: it takes a sequence of **tokens** (pieces of text) and returns a probability for every possible next token.

To generate text, we just do this in a loop:
1. Pick a next token using those probabilities
1. Stick it on the end of the input
1. Repeat until the model produces a special "stop" token

That's it. Everything else (chat, tool use, agents) is built on top of this loop.

### Let's run one

I'm going to load a small *open-weights* model, Qwen3-4B, released by Alibaba's Qwen team in July 2025. It has 4 billion parameters, squashed to 4 bits each, which makes it a 2.3 GB download that runs on my laptop.

Note, this is a *tiny* model compared to the ones behind Claude Code or Codex, and you'll see it make plenty of mistakes. That's actually quite useful for today.

In [ ]:
import mlx.core as mx
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

model, tokenizer = load("mlx-community/Qwen3-4B-Instruct-2507-4bit")

*Note, `mlx` only runs on Apple silicon Macs, so you don't need to run these cells yourself. [Ollama](https://ollama.com) or Hugging Face `transformers` let you do the same thing on other machines.*

Let's look at the probabilities it gives for the next token. This little function runs the model once and returns the `k` most likely next tokens as a pandas Series:

In [ ]:
def next_token_probs(text, k=8, temperature=1.0):
    ids = tokenizer.encode(text)
    logits = model(mx.array([ids]))[0, -1] / temperature  # one score per token in the vocabulary
    probs = pd.Series(mx.softmax(logits.astype(mx.float32)).tolist())
    top = probs.nlargest(k)
    top.index = [repr(tokenizer.decode([i])) for i in top.index]
    return top

In [ ]:
next_token_probs("The capital of France is").plot(kind='barh', figsize=(5, 3)).invert_yaxis()

Note, the model doesn't "look up" the answer, it just thinks " Paris" is the most likely continuation. About 2% of the time it would say " London"!

### Quick quiz

What is the most likely next token after:

> *"To read a CSV file in Python, most people use the"*

 - A. `' pandas'`
 - B. `' `'` (a backtick)
 - C. `' built'`
 - D. `' csv'`

In [ ]:
next_token_probs("To read a CSV file in Python, most people use the")

A backtick! The model has seen a lot of Markdown, where code like `` `pandas` `` is written in backticks. It's still going to say pandas, just formatted. The model is predicting *text*, not facts.

## Tokens

Models don't see letters or words, they see **tokens**: chunks of text from a fixed vocabulary, each with an integer ID. Qwen's vocabulary has about 151,000 of them.

In [ ]:
def show_tokens(text):
    ids = tokenizer.encode(text)
    print(len(ids), 'tokens:', ' | '.join(tokenizer.decode([i]) for i in ids))

show_tokens("Data science programming in the age of coding agents.")

Common words are usually a single token (including the space in front of them). Rarer words get split into pieces:

In [ ]:
show_tokens("df.groupby('pulocationid')['tip_pct'].mean()")

### Quick quiz

How many tokens is the word `"strawberry"`?

 - A. 1
 - B. 3
 - C. 10

In [ ]:
show_tokens("strawberry")
show_tokens(" strawberry")

Three on its own, but just one with a space in front. This is why LLMs famously struggle to count letters: the model never sees the individual letters, it sees token IDs.

Let's ask our model. We need a small helper to send it a chat message (we'll see what `apply_chat_template` does shortly):

In [ ]:
def ask(question, temperature=0.0, max_tokens=100):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    return generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens,
                    sampler=make_sampler(temp=temperature))

ask("How many r's are in strawberry? Just the number.")

Bigger models mostly get this right now, but often because they've been trained on this exact question. The underlying issue is the same.

### Numbers and other languages

Different models use different tokenizers. Qwen splits numbers into single digits, whereas OpenAI's `o200k_base` tokenizer groups them in threes (`123|456|789|0`), which affects how well a model does arithmetic:

In [ ]:
show_tokens("1234567890")
show_tokens("2026-10-12")

And not all languages are equal:

In [ ]:
sentences = {
    'English': ('The scientist studied the climate data.', 6),
    'German':  ('Die Wissenschaftlerin untersuchte Klimadaten.', 4),
    'Chinese': ('科学家研究了气候数据。', None),
    'Hindi':   ('वैज्ञानिक ने जलवायु डेटा का अध्ययन किया।', 7),
}
pd.DataFrame({lang: {'words': n, 'tokens': len(tokenizer.encode(s))}
              for lang, (s, n) in sentences.items()}).T

### Why do tokens matter?

Everything about using an LLM is measured in tokens rather than words or characters:
 - **Cost:** APIs charge per million input and output tokens
 - **Limits:** the context window (how much the model can "see" at once) is a number of tokens
 - **Speed:** output is generated one token at a time

A rough rule of thumb is ~1.3 tokens per English word, but it varies by model. Anthropic's newer Claude tokenizer, for example, produces about 30% more tokens for the same text than their older one. Code, numbers and non-English text use more tokens than you'd expect.

## Sampling and temperature

How do we pick the next token from those probabilities? We *sample* from them, and the **temperature** controls how adventurous that is: we divide the scores by the temperature before turning them into probabilities.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 2.5), sharex=True)
for ax, t in zip(axes, [0.5, 1.0, 2.0]):
    next_token_probs("My favourite colour is", temperature=t).plot(kind='barh', ax=ax)
    ax.invert_yaxis()
    ax.set_title(f'temperature = {t}')
fig.tight_layout()

Low temperatures sharpen the distribution towards the top token; high temperatures flatten it. Temperature 0 means always take the most likely token ("greedy" decoding).

In [ ]:
for t in [0.0, 0.0, 1.0, 1.0, 1.0]:
    print(f'temperature {t}:', ask("Suggest a name for a pet goldfish. Just the name.", temperature=t, max_tokens=10))

### Why does this matter?

 - The same prompt can give a different answer (or different code) each time. If something only works sometimes, run it again before trusting it.
 - You often can't turn this off: current Claude models, and the "reasoning" models from OpenAI and Google, don't let you set the temperature yourself.
 - So agent sessions are not reproducible in the way a script is. That's one reason to keep the *code* the agent writes, and its tests, rather than relying on re-running the conversation.

## Where do the probabilities come from?

Very briefly, modern LLMs are trained in stages:

1. **Pre-training:** predict the next token on a huge amount of text (a large part of the public internet, books, code...). This is where the knowledge comes from, and it's by far the most expensive step.
1. **Instruction tuning:** train on example conversations so it answers questions rather than just continuing text.
1. **Reinforcement learning:** reward good answers. Increasingly this uses tasks that can be checked automatically, like code that has to pass tests, which is a large part of why agents have got so much better at coding recently.

The training data stops at some point: the **knowledge cutoff**. Our Qwen model came out in July 2025; the current Claude models have a cutoff of June 2026.

### Quick quiz

What will our model say is the latest version of pandas?

 - A. 3.0
 - B. 2.2
 - C. 1.5.3
 - D. It will say it doesn't know

In [ ]:
ask("What is the latest version of pandas? Answer in one sentence.")

In [ ]:
ask("What's today's date? One sentence.")

Both answers are confident and wrong. pandas 2.0 came out in April 2023, and we're using pandas 3.0, which came out this year and changed some defaults (like the new `str` dtype and copy-on-write). And the model has no idea what today's date is unless someone tells it.

### Consequences for coding agents

 - Agents can write code for an **older version** of a library, or call functions that **don't exist** at all. One study of 576,000 generated code samples found that about 5% of the packages suggested by commercial models, and about 22% by open models, didn't exist ([Spracklen et al., 2025](https://arxiv.org/abs/2406.10279)). Attackers now register those made-up names ("slopsquatting").
 - The fix is mostly in the **harness**: tell the model the date and the versions installed, let it read documentation or search the web, and above all let it *run* the code and see the error.

## Open and closed models

| | Examples (Oct 2026) | How you use it |
|---|---|---|
| **Closed** | Claude (Anthropic), GPT (OpenAI), Gemini (Google) | Only through the company's API or apps; your data goes to them |
| **Open weights** | Qwen (Alibaba), DeepSeek, Gemma (Google), Mistral, gpt-oss (OpenAI), Kimi, GLM | Download the parameters and run them yourself, subject to the licence |
| **Fully open** | OLMo (Ai2) | Weights *and* training data, code and logs are published |

Note, most "open" models are only open weights: you can run them, but you can't see what they were trained on. And the licences vary from very permissive (Apache 2.0, MIT) to custom ones with restrictions.

### Why would you run your own?

 - **Data that can't leave your machine** (patient records, unpublished results, anything under a data use agreement)
 - **Cost** at scale, and no rate limits
 - **Reproducibility:** you can pin the exact weights, whereas API models change and get retired

But the best open models are now very large. The weights need roughly *(number of parameters) × (bytes per parameter)* of memory. Our 4B model at 4 bits (half a byte) is about 2 GB, which is about the biggest I can comfortably run on my MacBook Air. DeepSeek's latest has 1.6 trillion parameters.

### Quick quiz

Roughly how much memory do you need just to hold the weights of a 27-billion parameter model at 4 bits per parameter?

 - A. ~3.5 GB
 - B. ~14 GB
 - C. ~54 GB
 - D. ~108 GB

27 × 10⁹ × 0.5 bytes ≈ 13.5 GB, so **B**. That needs a well-specced workstation or a GPU server, and models this size (like Qwen3.6-27B or Gemma 4's 31B) are now quite capable. The trillion-parameter ones need a cluster of GPUs.

## A chat is just one long document

The model only ever does next-token prediction. So how does a "conversation" work? The **chat template** turns a list of messages into a single piece of text with special marker tokens:

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful data science assistant."},
    {"role": "user", "content": "How do I read a CSV file?"},
    {"role": "assistant", "content": "Use pd.read_csv('file.csv')."},
    {"role": "user", "content": "And an Excel file?"},
]
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

The model then just predicts what comes after `<|im_start|>assistant`, until it produces `<|im_end|>`.

### The model has no memory

Note, the model itself doesn't remember anything between calls. For every new message, the *whole* conversation so far gets sent again. The "memory" is entirely the harness's job.

Also note the **system prompt**: the user never sees it, but the model reads it first, every time. This is one of the main ways a harness steers a model.

### Quick quiz

You have a 20-turn conversation with an agent, and each turn adds 1,000 tokens (your message plus the reply). Roughly how many input tokens get sent to the model in total?

 - A. 20,000
 - B. 210,000
 - C. 400,000
 - D. 1,000,000

Turn 1 sends 1,000 tokens, turn 2 sends 2,000, ..., turn 20 sends 20,000. That adds up to 1,000 × (1 + 2 + ... + 20) = **210,000**. The total grows with the *square* of the conversation length (we'll come back to this in Module C).

Providers soften this with **prompt caching**: re-sending exactly the same beginning of a conversation is billed at a fraction of the price (around 5-10% of the normal input price for current Claude and OpenAI models). That makes it cheaper, but it still grows the same way.

### Context windows

The **context window** is the most tokens the model can take in at once: 262,000 for our Qwen model, and 1 million for the current Claude, GPT and Gemini models. That's a few thousand pages.

But you shouldn't fill it. Models get noticeably worse at using information as the context grows, even on simple tasks, and are worse at using information buried in the middle ([Liu et al., 2024](https://arxiv.org/abs/2307.03172); [Chroma, 2025](https://www.trychroma.com/research/context-rot)). Anthropic's own documentation calls this "context rot".

So a long, meandering agent session is not just more expensive, it gets *dumber*. Harnesses work around this, and so should you (more on Wednesday).

## Tool calling

So far the model can only produce text. How does an agent run code or edit files?

The harness describes the available **tools** in the prompt, and the model has been trained to write a special kind of text when it wants to use one. Here's a tool to multiply two numbers (the docstring and type hints become the description):

In [ ]:
def multiply(a: float, b: float):
    """
    A function that multiplies two numbers

    Args:
        a: The first number to multiply
        b: The second number to multiply
    """
    return a * b

messages = [{"role": "user", "content": "What is 12234585 times 48838483920?"}]
prompt = tokenizer.apply_chat_template(messages, tools=[multiply], add_generation_prompt=True)
print(tokenizer.decode(prompt))

The tool has been turned into a JSON description in the system prompt. Let's see what the model writes back:

In [ ]:
reply = generate(model, tokenizer, prompt=prompt, max_tokens=100)
print(reply)

The model **doesn't run anything**. It just writes text asking for a tool call. It's up to the harness to spot that, run the function, and paste the result back into the conversation as a new message.

(Which is just as well, since without the tool it gets the answer wrong:)

In [ ]:
print(ask("What is 12234585 times 48838483920? Just the number."))
print(12234585 * 48838483920)

## The agent loop

Putting that together gives us an **agent**: keep calling the model, running any tools it asks for, until it answers without asking for one.

Let's give our model a tool that runs Python, with the taxi data already loaded:

In [ ]:
import io, contextlib, json, re

taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv')

def run_python(code: str):
    """
    Run Python code and return whatever it prints. A pandas DataFrame called
    `taxis` (NYC taxi trips) is already loaded, and pandas is imported as pd.

    Args:
        code: The Python code to run
    """
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(code, {"taxis": taxis, "pd": pd})
    except Exception as e:
        return f"Error: {e!r}"
    return out.getvalue()

In [ ]:
def agent(question, tools, system=None, max_steps=6):
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": question})
    for step in range(max_steps):
        prompt = tokenizer.apply_chat_template(messages, tools=list(tools.values()), add_generation_prompt=True)
        reply = generate(model, tokenizer, prompt=prompt, max_tokens=500)
        messages.append({"role": "assistant", "content": reply})
        calls = re.findall(r"<tool_call>(.*?)</tool_call>", reply, re.S)
        if not calls:
            return reply  # no tool requested, so we're done
        for call in calls:
            call = json.loads(call)
            print(f"[step {step}] model asks to run:\n{call['arguments']['code']}")
            result = tools[call["name"]](**call["arguments"])  # the harness runs it
            print(f"[step {step}] harness sends back: {result!r}\n")
            messages.append({"role": "tool", "content": result})

That's about 15 lines. Real harnesses are much bigger, but this is the core of every coding agent.

### Quick quiz

What will our agent answer to *"Which payment type has the highest average tip?"*

Recall that `payment_type` is 1 for card and 2 for cash, and cash tips are always recorded as 0.

 - A. Card
 - B. Cash
 - C. It will crash
 - D. It will say it can't tell

In [ ]:
agent("Which payment type has the highest average tip in the taxi data?", tools={"run_python": run_python})

The code was fine, but it didn't `print` anything, so the tool sent back an empty string. The model then made up an answer that sounds plausible (and is wrong in a way you wouldn't spot without looking at the data).

### Improving the harness

Let's not touch the model at all. Instead we'll change the tool to behave like a Jupyter cell (show the value of the last line, and say so if there's no output), and add a system prompt with a short data dictionary:

In [ ]:
import ast

def run_python(code: str):
    """
    Run Python code and return the output, like a Jupyter cell. A pandas
    DataFrame called `taxis` (NYC taxi trips) is already loaded, and pandas
    is imported as pd.

    Args:
        code: The Python code to run
    """
    out = io.StringIO()
    env = {"taxis": taxis, "pd": pd}
    try:
        tree = ast.parse(code)
        last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        with contextlib.redirect_stdout(out):
            exec(compile(tree, "<agent>", "exec"), env)
            if last is not None:  # print the value of the last line, like Jupyter does
                print(repr(eval(compile(ast.Expression(last.value), "<agent>", "eval"), env)))
    except Exception as e:
        return f"Error: {e!r}"
    return out.getvalue() or "(the code ran but produced no output)"

SYSTEM = """You are a data analysis assistant. Answer questions using only the `taxis` DataFrame,
via the run_python tool. Never make up or simulate data.
payment_type: 1 = card, 2 = cash (cash tips are not recorded), 3 = no charge, 4 = dispute."""

In [ ]:
agent("Which payment type has the highest average tip in the taxi data?",
      tools={"run_python": run_python}, system=SYSTEM)

Same model, same question, correct answer. The only things we changed were what the tool sends back and the system prompt.

A 4B model is still a long way behind the frontier models (ask it something harder and it soon falls apart), but the harness clearly matters as much as the model!

## The anatomy of a harness

A production harness like Claude Code, Codex or Copilot has the same loop at its core, plus a lot of engineering around it:

| Part | What it does | In Claude Code |
|---|---|---|
| **System prompt** | How to behave, the date, the OS, the git status... | ~2,300 tokens |
| **Tools** | Read, edit and write files; run shell commands; search; fetch web pages | `Read`, `Edit`, `Write`, `Bash`, `Grep`, `Glob`, `WebFetch`, `WebSearch`, ... |
| **Project instructions** | Notes about *your* project, loaded into every session | `CLAUDE.md` (most other agents use `AGENTS.md`) |
| **Permissions** | Which tools can run without asking you | Modes from "ask for everything" to "auto" (a second model reviews each action) |
| **Context management** | Keeping the context small and relevant | Sub-agents with their own context, and summarising ("compacting") old history |
| **Extensions** | Connecting other tools and services | MCP servers, skills, hooks |

### Same model, different harness

You can see this in the benchmarks. On Terminal-Bench 2.1 (a set of realistic command-line tasks), the same model gets quite different scores depending on the harness it runs in. For example, GPT-5.5 scored 76% in one harness and 57% in another ([Vals.ai, Sep 2026](https://www.vals.ai/benchmarks/terminal-bench-2-1)).

So when you see "model X scores Y%", it's worth asking: in which harness, and with what settings?

### Open-source harnesses

Several good harnesses are open source, and most of them can use an open-weights model running on your own machine (e.g. through Ollama) instead of a paid API:

 - [OpenAI Codex CLI](https://github.com/openai/codex) (Apache 2.0)
 - [OpenCode](https://opencode.ai) (MIT)
 - [Aider](https://aider.chat), [Goose](https://github.com/aaif-goose/goose), [Cline](https://github.com/cline/cline), [Qwen Code](https://github.com/QwenLM/qwen-code), [Gemini CLI](https://github.com/google-gemini/gemini-cli)

*Tip: verified students get GitHub Copilot's student plan for free through [GitHub Education](https://github.com/settings/education/benefits), which may be useful for your project.*

## A real session, step by step

Let's go back to the tip analysis from Lecture 1. I set up a small project with the agent's original code in `tips.py`:

```python
def top_tip_zones(df, n=5):
    """Return the n pickup zones with the highest mean tip percentage."""
    df = df.copy()
    df["tip_pct"] = df["tip_amount"] / df["total_amount"] * 100
    return (df.groupby("pulocationid")["tip_pct"]
              .mean().sort_values(ascending=False).head(n))
```

and a `CLAUDE.md`:

```markdown
# Taxi tips analysis
- Python lives at ~/miniforge3/envs/dsc200-fa26/bin/python (pandas 3). pytest is not installed: write tests
  as plain `assert`s in `test_*.py` files and run them with that Python.
- Data: data/taxi_2019.csv, a 10,000-row sample of 2019 NYC yellow-taxi trips.
  payment_type: 1 = card, 2 = cash, 3 = no charge, 4 = dispute.
```

Then I asked Claude Code (Opus 5.5), allowing it to edit files and run Python, but nothing else without asking:

> *"The tip percentages from tips.py look off to me - I suspect cash trips are part of the problem. Can you fix top_tip_zones so the answer is meaningful, and add a test?"*

Claude Code can save every session as a log file, one JSON object per line. Let's read it in:

In [ ]:
session = [json.loads(line) for line in open('../data/claude_code_session.jsonl')]
init = session[0]
print(init['model'], init['permissionMode'])
print(len(init['tools']), 'tools:', ', '.join(init['tools']))

### What's in the context before I've typed anything?

Claude Code's `/context` command shows what the model will read before my prompt (this is for a fresh session in that folder):

| Category | Tokens |
|---|---|
| System prompt | 2.3k |
| Tool descriptions | 8.3k |
| Skills | 2.1k |
| Memory files (`CLAUDE.md`) | 0.2k |
| Tools that are only loaded when needed | (13.3k, not loaded) |
| **Total** | **12.9k of 1M** |

So roughly 13,000 tokens go in before it has done anything, and they get sent again on every step.

### The steps

Let's print each tool call and a short version of what came back:

In [ ]:
def short(text, n=300):
    text = text if isinstance(text, str) else json.dumps(text)
    return text if len(text) <= n else text[:n] + ' ...'

for event in session:
    if event['type'] == 'assistant':
        for block in event['message']['content']:
            if block['type'] == 'tool_use':
                args = block['input'].get('command') or block['input'].get('file_path')
                print(f"MODEL -> {block['name']}: {short(args, 200)}")
    elif event['type'] == 'user':
        for block in event['message']['content']:
            if block['type'] == 'tool_result':
                print(f"   HARNESS <- {'ERROR ' if block.get('is_error') else ''}{short(block['content'], 120)}\n")
    elif event['type'] == 'system' and event.get('subtype') == 'permission_denied':
        print(f"   PERMISSION DENIED: {event['message']}")

### What it did

1. **Looked around** first: listed the files, read `tips.py` and the first lines of the data. (The command "failed" only because there were no test files yet, and it carried on.)
1. **Explored the data** with two short Python scripts: tips by payment type, how many trips are behind the top zones, and the odd rows with negative totals.
1. **Wrote** the new `tips.py` and `test_tips.py`.
1. Tried to check that the tests would *fail* on the old code using `git stash`. I hadn't allowed git commands that change files, so **the harness blocked it**, and the model ran just the tests instead.
1. **Reported back** (and was honest that it hadn't run that check).

Note, it used the `CLAUDE.md` straight away: the right Python, no pytest, and what the payment types mean.

### The diff

```diff
-def top_tip_zones(df, n=5):
-    df = df.copy()
-    df["tip_pct"] = df["tip_amount"] / df["total_amount"] * 100
+CARD = 1
+
+def top_tip_zones(df, n=5, min_trips=30):
+    df = df[df["payment_type"] == CARD].copy()
+    df["pre_tip"] = df["total_amount"] - df["tip_amount"]
+    df = df[df["pre_tip"] > 0]
+    df["tip_pct"] = df["tip_amount"] / df["pre_tip"] * 100
+    zones = df.groupby("pulocationid")["tip_pct"].agg(["mean", "size"])
+    return (
+        zones.loc[zones["size"] >= min_trips, "mean"]
+             .sort_values(ascending=False)
+             .head(n)
+             .rename("tip_pct")
+    )
```

(docstring left out to fit on the slide)

### Quick quiz

Which of these did the agent decide **without being asked**?

 - A. Only using card trips
 - B. Ignoring zones with fewer than 30 trips
 - C. Changing the tip percentage to be relative to the fare *before* the tip
 - D. Both B and C

**D.** I suggested the cash problem; the other two were its own judgment calls. Both are reasonable, and to its credit it flagged them in its summary ("30 is my judgement call", "goes slightly beyond fix the cash problem"). But they change the answer, and it's your job to notice and decide.

### What did it cost?

Each step sent the whole conversation again. Let's plot the size of the context at each model call:

In [ ]:
calls = {}
for event in session:
    if event['type'] == 'assistant':
        calls[event['message']['id']] = event['message']['usage']  # one entry per model call
usage = pd.DataFrame(calls.values(), index=range(1, len(calls) + 1))
usage.columns = ['new', 'written to cache', 'read from cache']

ax = usage.plot(kind='bar', stacked=True, figsize=(6, 3), rot=0)
ax.set_xlabel('model call')
ax.set_ylabel('input tokens')

In [ ]:
result = session[-1]
total_in = sum(result['usage'][k] for k in ['input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens'])
print(f"{result['num_turns']} turns in {result['duration_ms'] / 1000:.0f} s")
print(f"input tokens: {total_in:,}, output tokens: {result['usage']['output_tokens']:,}")
print(f"actual cost: ${result['total_cost_usd']:.2f}")
print(f"without caching: ${total_in * 4 / 1e6 + result['usage']['output_tokens'] * 20 / 1e6:.2f}")

At Opus 5.5's prices ($4 per million input tokens, $20 per million output tokens), caching made this about 3-4 times cheaper. Most of the input is the same system prompt and tools being re-read every step. A long session on a real project can easily be a few dollars; we'll come back to cost in S13.

## Summary

| How it works | What it means for you |
|---|---|
| The model predicts the next token | It produces plausible text, not checked facts |
| It sees tokens, not letters or words | Cost, limits and some odd failures come from tokenization |
| Sampling is random | The same prompt can give different code; don't rely on re-running a conversation |
| Training stops at a knowledge cutoff | Expect outdated or made-up APIs; let it run code and read the docs |
| The whole conversation is resent each turn | Cost grows quickly with session length, and quality drops as the context fills up |
| The model only *asks* for tools | The harness decides what actually runs, so permissions matter |
| The harness shapes the behaviour | Same model, different results; `CLAUDE.md` / `AGENTS.md` are your lever |

Today, we covered the following topics:
 - Broadcasting by name, alignment and groupby in xarray
 - Next-token prediction, tokens and sampling
 - Training, knowledge cutoffs, and open versus closed models
 - Chat templates, context windows and tool calling
 - The agent loop, the anatomy of a harness and a real Claude Code session

On Wednesday (also on Zoom), we'll look at the agentic workflow: going from a vague request to a precise spec, and what difference that makes.